# recurrentppo_py12 — Colab 学習ノートブック (F-1)

ローカルにGPUが無いので、学習だけ Colab(GPU) で回すためのノートブック。

**手順**
1. ランタイム → ランタイムのタイプを変更 → **GPU** を選択
2. 下の「GPU確認」を実行
3. コード取得（**A: GitHub clone** か **B: Drive のフォルダ** のどちらか）
4. 依存インストール
5. **スモーク学習**で短時間動作確認 → 問題なければ **本学習**
6. チェックポイントを Drive に保存（再接続しても消えないように）

> ローカルの VSCode で編集 → GitHub へ push → ここで clone、が一番楽です。

## 0. GPU確認

In [ ]:
!nvidia-smi



## 1. コード取得（A か B のどちらか一方を実行）

### A) GitHub から clone
事前にローカルの変更を push しておくこと（`recurrentppo_py12` を含むコミット）。

In [1]:
# === 方法A: GitHub clone ===
REPO_URL = "https://github.com/Waipy252/resnet-dqn.git"
BRANCH = "main"

%cd /content
![ -d resnet-dqn ] && rm -rf resnet-dqn
!git clone --branch $BRANCH --depth 1 $REPO_URL
%cd /content/resnet-dqn/recurrentppo_py12_2
!pwd && ls


/content
Cloning into 'resnet-dqn'...
remote: Enumerating objects: 42, done.
remote: Counting objects: 100% (42/42), done.
remote: Compressing objects: 100% (36/36), done.
remote: Total 42 (delta 8), reused 24 (delta 2), pack-reused 0 (from 0)
Receiving objects: 100% (42/42), 9.22 MiB | 20.76 MiB/s, done.
Resolving deltas: 100% (8/8), done.
/content/resnet-dqn/recurrentppo_py12
/content/resnet-dqn/recurrentppo_py12
algo.py
calc_performance.py
config.py
data.py
docker-compose.yml
Dockerfile
docs
_eval_ensemble.py
_eval_one.py
main.py
nikkei_cp_1997-01-01_2024-01-01_146per_nenri_46_20260607.zip
notebooks
pyproject.toml
README.md
run_simulation.py
server.py
uv.lock
visualize.py


## 2. 依存インストール
Colab には CUDA 版 torch がプリインストール済みなので **torch は触らない**（壊さない）。
旧 `gym` も入れない（gymnasium に移行済み）。

In [2]:
!pip install -q \
    "stable-baselines3==2.8.0" \
    "sb3-contrib==2.8.0" \
    "gymnasium" "shimmy" \
    "yfinance" "pandas-datareader"
print('done')


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 187.5/187.5 kB 6.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 93.0/93.0 kB 9.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 952.1/952.1 kB 31.6 MB/s eta 0:00:00
done


## 3. スモーク学習（短時間で動作確認）
短い期間・少ステップで、データ取得→env→GPU学習→predict までを確認する。

In [ ]:
import numpy as np
import torch
from stable_baselines3.common.vec_env import DummyVecEnv
import config
from data import generate_env_data
from main import make_env
from algo import build_model, get_algo_class

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print('device =', device, '| algo =', get_algo_class().__name__)

# 短い期間でデータ取得（動作確認用）
df = generate_env_data('2015-01-01', '2024-01-01', ticker=config.TICKER)
print('data rows =', len(df))

env = DummyVecEnv([lambda: make_env(df)])
model = build_model(env, device, tensorboard_log=None, seed=0)
model.learn(total_timesteps=config.N_STEPS, progress_bar=True)

# RecurrentPPO の predict（LSTM状態は引数で受け渡す）
obs = env.reset()
state = None
episode_start = np.ones(1, dtype=bool)
action, state = model.predict(obs, state=state, episode_start=episode_start, deterministic=True)
print('smoke OK: device =', device, ' predicted action =', int(np.asarray(action).squeeze()))


## 4. 本学習（20シード ループ）
`main.py` をシード 0〜19 で順番に実行する。各シードは `config.TOTAL_TIMESTEPS` まで回り、
チェックポイントは `{model_name}_seed{N}_...` のプレフィックスで保存されるのでシード間で上書きされない。

途中であるシードが落ちても次のシードへ進む。GPU メモリ/速度を見たいときは別タブのセルで `!nvidia-smi` を実行。

In [ ]:
import subprocess, sys

SEEDS = range(20)  # シード 0〜19

for seed in SEEDS:
    print(f"\n{'='*30} seed {seed} {'='*30}", flush=True)
    ret = subprocess.run([sys.executable, "main.py", str(seed)])
    if ret.returncode != 0:
        print(f"seed {seed} が異常終了しました (returncode={ret.returncode})", flush=True)


In [ ]:
from google.colab import drive
drive.mount('/content/drive')


In [8]:
!python _eval_one.py


2026-06-09 14:01:39.807092: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
Gym has been unmaintained since 2022 and does not support NumPy 2.0 amongst other critical functionality.
Please upgrade to Gymnasium, the maintained drop-in replacement of Gym, or contact the authors of your software and request that they upgrade.
See the migration guide at https://gymnasium.farama.org/introduction/migration_guide/ for additional information.
/usr/local/lib/python3.12/dist-packages/pandas_datareader/compat/__init__.py:11: DeprecationWarning: distutils Version classes are deprecated. Use packaging.version instead.
  PANDAS_VERSION = LooseVersion(pd.__version__)
テストデータ取得中... 2022-06-01 → 2026-06-05
[*********************100%***********************]  1 of 1

## 5. チェックポイントを Drive に保存
（方法A で clone した場合、保存先はランタイム上なので消える。Drive にコピーして永続化する。）

In [ ]:
import glob, os, shutil
try:
    from google.colab import drive
    drive.mount('/content/drive')
except Exception as e:
    print('drive mount skip:', e)

DST = '/content/drive/MyDrive/resnet-dqn-models'
os.makedirs(DST, exist_ok=True)
files = sorted(set(glob.glob('nikkei_rppo_*.zip')))
for f in files:
    shutil.copy(f, DST)
print(f'copied {len(files)} files to {DST}')


## 6. （任意）評価
`_eval_one.py` が複数OOS窓（コロナ/軟調/bull）のデータを取得し、各チェックポイントを
バックテストして窓ごとに B&H と比較する。重いので動作確認だけなら飛ばしてよい。


In [ ]:
!python _eval_one.py
